In [ ]:
import torch
import matplotlib.pyplot as plt

x = torch.tensor([[1.0],[2.0],[3.0],[4.0]])
y = torch.tensor([[2.0],[4.0],[6.0],[8.0]])  # 정답: y = 2x

w = torch.zeros(1, requires_grad=True)
optimizer = torch.optim.SGD([w], lr=0.01)
losses = []
for epoch in range(200):
    pred = x * w
    loss = ((pred - y)**2).mean()
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    losses.append(loss.item())

print("학습된 w:", w.item())
plt.plot(losses); plt.xlabel("epoch"); plt.ylabel("Loss"); plt.title("Learning = minimizing loss")
plt.show()


학습된 w: 1.9999996423721313


- 곡선이 내려가는 모습 = "오차가 줄어드는 것" = "학습되는 것"

---

## 본 실습 — Hugging Face 파이프라인

In [ ]:
!pip install transformers -q


## 1. 감정 분석 (가장 가벼움)

비유: "이미 영어 학원 다 다닌 학생을 한 줄 명령으로 데려오는 셈."

In [ ]:
from transformers import pipeline
clf = pipeline("sentiment-analysis")
print(clf("I love this class!"))
print(clf("This homework is a nightmare."))


[{'label': 'POSITIVE', 'score': 0.9998816251754761}]
[{'label': 'NEGATIVE', 'score': 0.9993976354598999}]


**학생 활동**: 본인 문장 5개를 만들어 결과 공유.

In [ ]:
# 학생 본인 문장 5개
for s in [
    "오늘 아침 수업 있네",
    "졸리다",
    "수업 언제 끝나지?",
    "야르~",
    "오 신기하다 ㅋㅋ",
]:
    print(s, "→", clf(s)[0])


오늘 아침 수업 있네 → {'label': 'POSITIVE', 'score': 0.8404001593589783}
졸리다 → {'label': 'POSITIVE', 'score': 0.6479342579841614}
수업 언제 끝나지? → {'label': 'NEGATIVE', 'score': 0.8190799951553345}
야르~ → {'label': 'POSITIVE', 'score': 0.8849546313285828}
오 신기하다 ㅋㅋ → {'label': 'POSITIVE', 'score': 0.8566548824310303}


> **모델이 틀리는 경우 일부러 보여주기** — 9주차 환각(Slide 22) 복기.

## 2. 한·영 번역 (시간 여유 시)

비유: "한국어 학원과 영어 학원을 동시에 다닌 학생을 한 줄로 데려오는 셈."
Block 3에서 본 토큰화가 실제로 어떻게 쓰이는지 자연스럽게 이어진다.


### 2-1. [새로운 예제] 한-영 번역 통합 코드
기존 코드에서 오류가 발생한다면 아래의 통합 코드를 실행해 보세요.

In [ ]:
# 1. 필요한 라이브러리 설치
!pip install sacremoses -q

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 2. 모델 및 토크나이저 로드
model_name = "Helsinki-NLP/opus-mt-ko-en"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def translate_ko_to_en(text):
    # 입력 텍스트 토큰화
    inputs = tokenizer(text, return_tensors="pt", padding=True)
    # 모델을 이용해 번역 수행 (pipeline 내부 동작을 직접 구현)
    with torch.no_grad():
        translated_tokens = model.generate(**inputs)
    # 결과 토큰을 텍스트로 복원
    return tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]

# 3. 테스트 실행
test_sentences = [
    "코딩은 정말 재미있어요.",
    "오늘 날씨가 참 좋네요.",
    "Hugging Face를 활용한 번역 실습입니다."
]

for text in test_sentences:
    result = translate_ko_to_en(text)
    print(f"입력: {text}")
    print(f"번역: {result}\n")

> **모델이 어색하게 번역하는 경우 일부러 보여주기** — 고유명사·신조어·반어법은 약함. "학습한 적 없는 단어는 못 한다"는 9주차 한계 슬라이드 복기.

In [ ]:
# 모델의 한계를 보여주는 예시 (신조어, 반어법, 고유명사)
limit_test_sentences = [
    "이거 실화냐?", # 신조어/슬랑
    "그쪽 실력이 정말 대~단하시네요.", # 반어법(비꼼)
    "너 T발 C야?", # 최근 유행하는 밈/신조어
    "갓생 살기 프로젝트를 시작했다.", # 합성 신조어
]

print("--- 모델의 한계 테스트 ---")
for text in limit_test_sentences:
    result = translate_ko_to_en(text)
    print(f"입력: {text}")
    print(f"번역: {result}\n")

## 마무리

9주차 어텐션(Slide 12) 비유 재인용:
> "지금 모델 안에서 단어들이 서로 눈치 보면서 어디에 집중할지 결정하고 있는 겁니다. 우리가 한 줄 명령으로 그걸 굴린 거예요."